# Denoising: 10k vs 50k crops — same held-out val set

Trains the model **twice** on the same dataset folder — full **50k** set first (run this one alone first — it's the one that has diverged with NaN before, so it's the fastest way to know if the current fix holds), then capped to **10k** training crops, sharing **one identical validation set** so the two are directly comparable. Then regenerates a prediction-evolution figure from per-epoch checkpoints.

### Steps
1. **Upload the 50k dataset once:** on your Mac it's `data/crops50k.zip`. Kaggle → **Create → New Dataset** → upload that zip (auto-extracts) → name it e.g. `document-crops-50k`.
2. **+ Add Input → Datasets** → attach it. *(Attached mid-session? Restart the session.)*
3. **Settings:** Accelerator → **GPU T4**, Internet → **On**.
4. **Run the 50k cell alone first** and watch for NaN before running anything else.
5. Once clean, run the rest (**Run All**, or **Save Version → Save & Run All (Commit)** to survive closing the tab).
6. Download **`results.zip`** from the **Output** tab.

Only one upload — the 10k run is the same folder with `data.max_train_samples=10000`, so its val set is identical to the 50k run's and its train set is a strict subset.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator to GPU")

In [ ]:
REPO_URL = "https://github.com/jere1882/DocDenoising.git"
!rm -rf DocDenoising
!git clone --depth 1 {REPO_URL}
%cd DocDenoising
!pip install -q -e .
print("install done")

In [ ]:
import glob, os
pngs = glob.glob("/kaggle/input/**/*.png", recursive=True)
assert pngs, "No .png under /kaggle/input — attach the 50k dataset and restart the session."
CLEAN_DIR = os.path.dirname(pngs[0])
EPOCHS = 2  # smoke-test value — confirm no NaN in val_psnr before raising this
print(f"CLEAN_DIR = {CLEAN_DIR}  ({len(pngs)} crops), EPOCHS = {EPOCHS}")

In [ ]:
# --- Run 1: full (~50k) training crops — this is the one that has diverged before ---
# Run this cell ALONE first and watch val_loss/val_psnr in the live output — if either
# prints "nan" at any epoch, stop; no need to also burn time on the 10k cell below.
# gradient_clip_val=1.0 is the trainer default (didn't fix a prior NaN by itself —
# see notes at the bottom — but leaving it on doesn't hurt).
!denoising-train data.clean_dir={CLEAN_DIR} \
    save_all_epochs=true hydra.run.dir=/kaggle/working/run_50k \
    trainer.max_epochs={EPOCHS} +trainer.precision=16-mixed logger=csv

In [ ]:
# --- Run 2: 10k training crops, SAME val set — only run after confirming run_50k above is clean ---
!denoising-train data.clean_dir={CLEAN_DIR} data.max_train_samples=10000 \
    save_all_epochs=true hydra.run.dir=/kaggle/working/run_10k \
    trainer.max_epochs={EPOCHS} +trainer.precision=16-mixed logger=csv

In [ ]:
# --- Regenerate prediction evolution from per-epoch checkpoints ---
!denoising-viz-evolution --run-dir /kaggle/working/run_10k
!denoising-viz-evolution --run-dir /kaggle/working/run_50k
from IPython.display import Image, display
display(Image("/kaggle/working/run_10k/evolution.png"))
display(Image("/kaggle/working/run_50k/evolution.png"))

In [ ]:
# --- Which is better? val PSNR on the shared held-out set (same crops, same noise) ---
import glob, pandas as pd, matplotlib.pyplot as plt

def curve(run):
    f = glob.glob(f"{run}/logs/denoising/version_*/metrics.csv")[0]
    df = pd.read_csv(f)
    return df.dropna(subset=["val_psnr"]).groupby("epoch")["val_psnr"].last()

c10, c50 = curve("/kaggle/working/run_10k"), curve("/kaggle/working/run_50k")
print(f"best val PSNR — 10k: {c10.max():.3f} dB   50k: {c50.max():.3f} dB   delta: {c50.max()-c10.max():+.3f} dB")

plt.figure(figsize=(6, 4))
plt.plot(c10.index, c10.values, "-o", label="10k")
plt.plot(c50.index, c50.values, "-o", label="50k")
plt.xlabel("epoch"); plt.ylabel("val PSNR (dB)"); plt.grid(alpha=0.3); plt.legend()
plt.title("val PSNR vs epoch (shared val set)")
plt.savefig("/kaggle/working/val_psnr.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# --- Bundle results for the Output tab ---
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "/kaggle/working")
print("wrote /kaggle/working/results.zip")

### Notes
- **Same val set, guaranteed.** Both runs use the same folder + seed, and `max_train_samples` caps only the *training* set, so the validation split is identical and the 10k train set is a subset of the 50k one.
- **Step-vs-epoch caveat.** Both use the same `EPOCHS`, so the 50k run does ~5× more gradient updates — a win could be "more data" or "more training." To equalize compute, cap both by steps: add `trainer.max_steps=<N> trainer.max_epochs=-1`.
- **Regenerating the figure later.** Checkpoints (`epoch=00.ckpt`, …) live under each `run_*/checkpoints/`. Re-run `denoising-viz-evolution --run-dir <run>` any time — e.g. `--num-samples 8` for more crops.
- **NaN history**: a prior run with `+trainer.precision=16-mixed` produced NaN val_psnr from epoch 3 onward (10k) / epoch 0 onward (50k) — fp16 overflow, most likely inside BatchNorm's forward-pass variance computation (a batch's aggregate variance can overflow fp16 even with no single extreme pixel). Adding `gradient_clip_val=1.0` (now the trainer default) did NOT fix it in a repeat run — 50k still went NaN at epoch 0 — which makes sense, since clipping only touches backward-pass gradients, not this forward-pass statistic. A local Mac (MPS, not CUDA) repro of the same run_50k config got through 2 clean epochs with no NaN, suggesting this may be specific to the CUDA/T4 fp16 BatchNorm path rather than the data itself. Run the 50k cell alone first and confirm `val_psnr` stays a real number before trusting anything longer.
- **Runtime — measured, not guessed** (single T4, `unet`, 512px, bs 8, with mixed precision): ~1.7 it/s. 50k run ≈ 55 min/epoch (5625 steps). 10k run ≈ 11 min/epoch (1125 steps). At `EPOCHS=2`: ~110 min + ~22 min. Raise `EPOCHS` only after confirming no NaN, and budget accordingly against your weekly GPU quota — a full 10-epoch run of both is ~11-13 hours.